# Risk Actions: Pre-LLM vs Post-LLM Explanations

This notebook is a learning notebook. The goal is to make the LLM layer visible instead of hidden inside the dashboard.

We will compare:

- **Pre-LLM explanation**: deterministic text created from the `HoldingActionRecommendation` object.
- **Post-LLM explanation**: plain-English rewrite generated by Ollama from the same deterministic evidence.

Important rule for this project:

**Math decides. Evidence supports. LLM explains. UI earns trust.**

## Agentic Lifecycle For This Notebook

Think of the LLM explanation layer as an agentic step with guardrails:

1. **Observe**: Load the saved portfolio diagnosis and action recommendations.
2. **Diagnose**: Identify the holdings that already passed the rule-based sell/trim gate.
3. **Plan**: Build a compact evidence payload for each holding.
4. **Act**: Ask Ollama to rewrite the evidence in plain English.
5. **Verify**: Compare deterministic text vs LLM text side by side.
6. **Explain**: Confirm that the LLM did not change the action, amount, ranking, or confidence.

The LLM is not a portfolio manager here. It is a communication layer.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

# Make the notebook robust whether you open it from repo root or data_pipeline.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "data_pipeline" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DIAGNOSIS_DIR = PROJECT_ROOT / "data" / "processed" / "diagnosis"
DIAGNOSIS_DIR

In [ ]:
from portfolio_analyzer.diagnosis import portfolio_risk_diagnosis_from_saved_artifacts
from portfolio_analyzer.app import (
    _risk_action_deterministic_explanation,
    _risk_action_llm_payload,
    generate_risk_action_llm_explanations,
    preferred_risk_action_llm_model,
    percent_display,
    money_text,
)

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)
actionable = [item for item in diagnosis.holding_action_recommendations if item.is_actionable]

print(f"Loaded diagnosis run: {diagnosis.run_id}")
print(f"Actionable Risk Actions: {len(actionable)}")
print(f"Available Ollama model for explanations: {preferred_risk_action_llm_model()}")

## 1. Pre-LLM Explanations

This table is the pre-LLM baseline. It is created directly from the deterministic action object.

If the app had no LLM available, this is the fallback explanation style users would still see.

In [ ]:
def explanation_to_row(item, explanation, prefix):
    return {
        "Ticker": item.ticker,
        "Action": item.recommendation_label,
        "Value to sell": money_text(item.value_to_sell),
        "Reduction %": percent_display(item.position_reduction_pct),
        "Confidence": item.confidence_band,
        "Source": explanation.get("_source", "Unknown"),
        f"{prefix} headline": explanation.get("headline", ""),
        f"{prefix} summary": explanation.get("plain_english_summary", ""),
        f"{prefix} why now": explanation.get("why_now", ""),
        f"{prefix} context": explanation.get("external_context", ""),
        f"{prefix} improves": explanation.get("what_improves", ""),
        f"{prefix} watchout": explanation.get("watchout", ""),
    }

pre_llm_rows = [
    explanation_to_row(item, _risk_action_deterministic_explanation(item), "Pre-LLM")
    for item in actionable
]

pre_llm_df = pd.DataFrame(pre_llm_rows)
pre_llm_df

## 2. Evidence Payload Sent To The LLM

This is the most important trust-building cell.

The LLM is not allowed to look around freely. It only receives this compact JSON payload, which includes:

- the already-decided action and amount
- recent performance versus the S&P 500
- portfolio-risk evidence
- external context from company reports, news, fundamentals, and macro signals
- existing deterministic explanations and guardrails

In [ ]:
if actionable:
    first_action = actionable[0]
    payload = _risk_action_llm_payload(diagnosis, first_action)
    print(f"Showing LLM payload for: {first_action.ticker}")
    print(json.dumps(payload, indent=2)[:8000])
else:
    print("No actionable holdings were available in the latest diagnosis.")

## 3. Post-LLM Explanations

This cell calls Ollama and asks it to rewrite the deterministic evidence in plain English.

The call is cached by evidence payload, so repeated runs should be faster unless the evidence changes.

If Ollama is not running, the notebook will still return the rule-based fallback. Check the `Source` column to see what happened.

In [ ]:
# Set this to False if you want to inspect the notebook without calling Ollama.
RUN_OLLAMA = True

if RUN_OLLAMA:
    post_explanations = generate_risk_action_llm_explanations(diagnosis, actionable)
else:
    post_explanations = {
        item.ticker: _risk_action_deterministic_explanation(item)
        for item in actionable
    }

post_llm_rows = [
    explanation_to_row(item, post_explanations.get(item.ticker, {}), "Post-LLM")
    for item in actionable
]

post_llm_df = pd.DataFrame(post_llm_rows)
post_llm_df

## 4. Side-by-Side Comparison

Use this table to learn what the LLM changed.

Good LLM behavior:

- simpler wording
- better connection between performance, risk, and external evidence
- clearer watchouts
- no change to action, amount, ranking, or confidence

Bad LLM behavior:

- inventing facts
- changing the recommendation
- making the action sound more certain than the evidence supports
- vague language that could apply to any stock

In [ ]:
comparison_rows = []
for item in actionable:
    pre = _risk_action_deterministic_explanation(item)
    post = post_explanations.get(item.ticker, {})
    comparison_rows.append(
        {
            "Ticker": item.ticker,
            "Action stays fixed": item.recommendation_label,
            "Amount stays fixed": money_text(item.value_to_sell),
            "Post source": post.get("_source", "Unknown"),
            "Pre headline": pre.get("headline", ""),
            "Post headline": post.get("headline", ""),
            "Pre why now": pre.get("why_now", ""),
            "Post why now": post.get("why_now", ""),
            "Post external context": post.get("external_context", ""),
            "Post watchout": post.get("watchout", ""),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 5. Audit Checklist

Use this checklist when reviewing LLM output:

- Did the **action label** stay the same?
- Did the **sell amount** stay the same?
- Did the LLM explain **why now** in regular language?
- Did the LLM mention external evidence only when evidence exists?
- Did the LLM keep uncertainty visible?
- Would a non-technical investor understand the recommendation in 10 seconds?

If the answer is no, improve the prompt or the evidence payload. Do not let the LLM override the deterministic object.

In [ ]:
audit_rows = []
for item in actionable:
    post = post_explanations.get(item.ticker, {})
    source = post.get("_source", "Unknown")
    audit_rows.append(
        {
            "Ticker": item.ticker,
            "Action fixed by Python": item.recommendation_label,
            "Amount fixed by Python": money_text(item.value_to_sell),
            "Explanation source": source,
            "LLM actually used?": source.startswith("Ollama:"),
            "Has why-now sentence?": bool(post.get("why_now")),
            "Has watchout?": bool(post.get("watchout")),
            "Has confidence note?": bool(post.get("confidence_note")),
        }
    )

pd.DataFrame(audit_rows)

## What You Should Learn From This Notebook

This is the pattern we will reuse across the project:

1. Build a deterministic object first.
2. Create a compact evidence payload from that object.
3. Ask the LLM to explain, not decide.
4. Label the output source clearly.
5. Keep fallback behavior if the LLM is unavailable.
6. Compare pre-LLM and post-LLM output before trusting it in the dashboard.

That is the core agentic lifecycle in this app.